# RU OCR: export, ONNX, inference check, predictions on stroyinf v2

1. **Export**: training checkpoint -> inference model (`tools/export_model.py`: `inference.json`,
   `inference.pdiparams`, `inference.yml`).
2. **ONNX** for CPU users (`paddlex --paddle2onnx` in a Python 3.12 venv: `inference.onnx` + `inference.yml`).
3. **Check**: the same lines through Paddle on GPU, ONNX Runtime on CPU and OpenVINO on CPU (PaddleX `create_model`;
   ONNX Runtime with the CPU or the OpenVINO execution provider).
4. **Predictions** on stroyinf v2 (before 1992) with the exported model on both GPUs (or OpenVINO on CPU); lines
   where the prediction differs from the text-layer label go to `review.tsv` for a manual check (stage 7, label
   cleaning). Every `SAVE_EVERY` lines go to `preds_partial.tsv`, which is also uploaded every 10 min to the private
   dataset `PREDS_DATASET` (needs the `KAGGLE_API_TOKEN` secret). A session that hits the time limit or crashes is
   resumed by attaching `PREDS_DATASET` (or the session's output) as Input and running again -- the lines already in
   its `preds_partial.tsv` are skipped.

**Input**: `ru-ocr-stroyinf` (crops), `ru-ocr-labels`, `ru-ocr-checkpoints-ft3` (weights).
**Settings**: Accelerator GPU T4 x2, Internet on, Add-ons -> Secrets: `KAGGLE_API_TOKEN` (without it nothing is
uploaded, the predictions are only in the output). Save & Run All (Commit).
Without a GPU: `DEVICE = "cpu"`, Accelerator None -- section 4 runs on OpenVINO (CPU sessions are outside the GPU
quota); `PRED_LIMIT = 20` for a quick test run.

**Output** (`/kaggle/working`): `model/inference` (Paddle), `model/onnx` (ONNX), `preds_all.tsv.gz` (every line:
path, label, pred, score, dist, dist_ns = edits ignoring spaces), `review.tsv` (lines that differ in
characters, the largest disagreements first), `review_spaces.tsv` (lines that differ only in spaces),
`preds_partial.tsv` (index, path, pred, score; saved as the predictions go).

In [ ]:
CKPT = "best_accuracy"   # checkpoint prefix in CKPT_DATASET: best_accuracy or latest
CKPT_DATASET = "ru-ocr-checkpoints-ft3"
CONFIG = "ru_ocr/configs/ru_PP-OCRv6_small_rec.yml"
MODEL_NAME = "PP-OCRv6_small_rec"
LABEL_FILES = ["stroyinf_all_train_w_h.txt", "stroyinf_all_val_w_h.txt"]  # stroyinf v2: 580 800 + 28 913 lines
DEVICE = "cpu"           # "cpu": no GPU -- CPU paddle for export and check, OpenVINO for the predictions
PRED_LIMIT = None        # first lines predicted on stroyinf (None = all; 20 for a quick test run)
SAVE_EVERY = 2000        # predictions appended to preds_partial.tsv every SAVE_EVERY lines (per device)
PREDS_DATASET = "vladislavershov/ru-ocr-stroyinf-preds"  # preds_partial.tsv uploaded here every 10 min; None: no upload
CHECK_LINES = 3          # lines for the backend check
BATCH = 64
MODEL_DIR = "/kaggle/working/model"
import os, subprocess, sys, time
if DEVICE == "gpu":
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

## 0. Paddle GPU, PaddleX, code

In [ ]:
import shlex, signal, threading


def sh(cmd, timeout=1200, attempts=3):
    """Run a shell command with its output in the cell. Some installs stall forever with an idle CPU: a run is
    killed with its whole process group after `timeout` seconds and retried -- the same as pressing Stop by hand
    (Stop kills a hung `!` command and the cell goes on)."""
    for attempt in range(1, attempts + 1):
        print(time.strftime("%H:%M:%S"), f"[{attempt}/{attempts}]", cmd, flush=True)
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                start_new_session=True)
        timer = threading.Timer(timeout, os.killpg, (proc.pid, signal.SIGKILL))
        timer.start()
        for line in proc.stdout:
            print(line, end="", flush=True)
        hung = not timer.is_alive()
        timer.cancel()
        if proc.wait() == 0:
            return
        print(time.strftime("%H:%M:%S"), f"hung for {timeout} s, killed" if hung else f"exit code {proc.returncode}", flush=True)
    raise RuntimeError(f"failed {attempts} times: {cmd}")


if DEVICE == "gpu":
    paddle_pkg = "paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/"
else:
    paddle_pkg = "paddlepaddle==3.3.1"
PIP = f"{shlex.quote(sys.executable)} -m pip install --progress-bar off --timeout 120 --retries 10"
sh(f"{PIP} {paddle_pkg}", attempts=5)
# onnxruntime-openvino: ONNX Runtime with the OpenVINO execution provider (replaces plain onnxruntime;
# the PaddleX hpi-cpu plugin, ultra-infer, has no Python 3.13 build)
sh(f"{PIP} 'paddlex[ocr-core]>=3.7,<3.8' onnxruntime-openvino")
# paddle2onnx has wheels only for Python <= 3.12 (Kaggle runs 3.13): the ONNX conversion runs in its own venv
sh(f"{PIP} uv")
sh("uv venv --clear --seed --python 3.12 /tmp/py312")  # --clear: rerunnable; --seed: pip inside, paddlex --install uses it
sh("uv pip install --python /tmp/py312/bin/python paddlepaddle==3.3.1 'paddlex[ocr-core]>=3.7,<3.8'")
sh("/tmp/py312/bin/paddlex --install paddle2onnx")
import onnxruntime
print("ONNX Runtime providers:", onnxruntime.get_available_providers())

In [ ]:
%cd /kaggle/working
!rm -rf PaddleOCR && git clone -q --depth 1 --branch feat/ru-ppocrv6-small-training https://github.com/ErshovVE/PaddleOCR.git
%cd /kaggle/working/PaddleOCR
!git log --oneline -1
sh(f"{PIP} -r requirements.txt")

## Data, labels, weights
The label paths start with `stroyinf/<folder>/crops/...`: the folders that hold `crops/` are linked into
`/tmp/ru_root/stroyinf`.

In [ ]:
import pickle

ROOT = "/tmp/ru_root"
os.makedirs(os.path.join(ROOT, "stroyinf"), exist_ok=True)
labels_dir, weights, partials = None, [], []
for top, dirs, files in os.walk("/kaggle/input"):
    if "crops" in dirs:
        dirs.remove("crops")
        if os.path.basename(os.path.dirname(top)) == "stroyinf":
            link = os.path.join(ROOT, "stroyinf", os.path.basename(top))
            if not os.path.exists(link):
                os.symlink(top, link)
    if LABEL_FILES[0] in files:
        labels_dir = top
    if CKPT + ".pdparams" in files and CKPT_DATASET in top.split(os.sep):
        weights.append(os.path.join(top, CKPT))
    if "preds_partial.tsv" in files:  # output of an earlier session of this notebook, added as Input
        partials.append(os.path.join(top, "preds_partial.tsv"))
assert labels_dir and len(weights) == 1, f"attach ru-ocr-labels and {CKPT_DATASET}: {weights}"
WEIGHTS = weights[0]
states = pickle.load(open(WEIGHTS + ".states", "rb"), encoding="latin1")
print("stroyinf folders:", len(os.listdir(os.path.join(ROOT, "stroyinf"))), "labels:", labels_dir)
print("earlier predictions:", partials)
print("weights:", WEIGHTS, "global step", states.get("global_step"), "val acc", states.get("best_model_dict", {}).get("acc"))

## 1. Export: checkpoint -> inference model

In [ ]:
!python tools/export_model.py -c {CONFIG} -o Global.pretrained_model={WEIGHTS} Global.save_inference_dir={MODEL_DIR}/inference 2>&1 | grep -v -i warn | tail -3
!ls -la {MODEL_DIR}/inference

## 2. ONNX for CPU

In [ ]:
# PATH: paddlex looks for the paddle2onnx executable of the venv
env = dict(os.environ, PATH="/tmp/py312/bin:" + os.environ["PATH"])
subprocess.run(["paddlex", "--paddle2onnx", "--paddle_model_dir", f"{MODEL_DIR}/inference",
                "--onnx_model_dir", f"{MODEL_DIR}/onnx"], env=env, check=True)
!ls -la {MODEL_DIR}/onnx

## 3. Check: Paddle (GPU, or CPU when DEVICE = "cpu"), ONNX Runtime CPU, OpenVINO CPU
The same `CHECK_LINES` lines through each backend; the texts must match.

In [ ]:
from paddlex import create_model

with open(os.path.join(labels_dir, LABEL_FILES[1]), encoding="utf-8") as f:
    check = [line.rstrip("\n").split("\t")[:2] for _, line in zip(range(CHECK_LINES), f)]
check_paths = [os.path.join(ROOT, p) for p, _ in check]
backends = {
    f"paddle {DEVICE}": dict(model_dir=f"{MODEL_DIR}/inference", device=DEVICE + (":0" if DEVICE == "gpu" else "")),
    "onnxruntime cpu": dict(model_dir=f"{MODEL_DIR}/onnx", device="cpu", engine="onnxruntime",
                            engine_config={"providers": ["CPUExecutionProvider"]}),
    "openvino cpu": dict(model_dir=f"{MODEL_DIR}/onnx", device="cpu", engine="onnxruntime",
                         engine_config={"providers": ["OpenVINOExecutionProvider"], "provider_options": {"device_type": "CPU"}}),
}
texts = {}
for name, kwargs in backends.items():
    model = create_model(model_name=MODEL_NAME, **kwargs)
    start = time.time()
    res = list(model.predict(check_paths, batch_size=1))
    texts[name] = [r["rec_text"] for r in res]
    print(f"--- {name}: {time.time() - start:.2f} s")
    for (_, label), r in zip(check, res):
        print(f"  {r['rec_score']:.3f}  {r['rec_text']}  | label: {label}")
    del model
ref_name = f"paddle {DEVICE}"
for name, t in texts.items():
    print(name, f"same as {ref_name}" if t == texts[ref_name] else f"DIFFERS from {ref_name}")

## 4. Predictions on stroyinf v2: one predictor per GPU
Each GPU gets its own Paddle predictor (`create_model(device="gpu:N")`) in a thread and takes every second line.
With `DEVICE = "cpu"`: one OpenVINO predictor on the ONNX model (the backend checked in section 3).
Every `SAVE_EVERY` lines of a device are appended to `/kaggle/working/preds_partial.tsv`; lines found in an earlier
`preds_partial.tsv` (Input) or already in the working one are skipped. A background thread
(`ru_ocr/tools/ckpt_uploader.py`) uploads `preds_partial.tsv` to `PREDS_DATASET` every 10 min and once at the end.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

rows = []
for name in LABEL_FILES:
    with open(os.path.join(labels_dir, name), encoding="utf-8") as f:
        rows += [line.rstrip("\n").split("\t")[:2] for line in f if line.strip()]
if PRED_LIMIT:
    rows = rows[:PRED_LIMIT]
print(len(rows), "lines")

PARTIAL = "/kaggle/working/preds_partial.tsv"
preds = {}
for path in partials + ([PARTIAL] if os.path.exists(PARTIAL) else []):
    with open(path, encoding="utf-8") as f:
        for line in f:
            i, p, text, score = line.rstrip("\n").split("\t")
            i = int(i)
            if i < len(rows) and rows[i][0] == p:  # same line of the same label files
                preds[i] = (text, float(score))
with open(PARTIAL, "w", encoding="utf-8") as f:  # everything done so far, so this session's output is complete
    f.writelines(f"{i}\t{rows[i][0]}\t{t}\t{s}\n" for i, (t, s) in sorted(preds.items()))
print(len(preds), "lines predicted earlier")

if DEVICE == "gpu":
    predictors = [dict(model_dir=f"{MODEL_DIR}/inference", device=d) for d in ("gpu:0", "gpu:1")]
else:
    predictors = [dict(model_dir=f"{MODEL_DIR}/onnx", device="cpu", engine="onnxruntime",
                       engine_config={"providers": ["OpenVINOExecutionProvider"], "provider_options": {"device_type": "CPU"}})]
todo = [i for i in range(len(rows)) if i not in preds]
progress = [0] * len(predictors)
write_lock = threading.Lock()


def run(k):
    model = create_model(model_name=MODEL_NAME, **predictors[k])
    part = todo[k :: len(predictors)]
    for start in range(0, len(part), SAVE_EVERY):
        idx = part[start : start + SAVE_EVERY]
        res = model.predict([os.path.join(ROOT, rows[i][0]) for i in idx], batch_size=BATCH)
        chunk = [(i, r["rec_text"], float(r["rec_score"])) for i, r in zip(idx, res)]
        with write_lock, open(PARTIAL, "a", encoding="utf-8") as f:
            f.writelines(f"{i}\t{rows[i][0]}\t{t}\t{s}\n" for i, t, s in chunk)
        for i, t, s in chunk:
            preds[i] = (t, s)
        progress[k] += len(idx)


uploader = None
if PREDS_DATASET:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
        sh(f"{PIP} --upgrade kaggle")
        sys.path.insert(0, "ru_ocr/tools")
        from ckpt_uploader import CheckpointUploader
        uploader = CheckpointUploader("/kaggle/working", "/tmp/preds_upload", PREDS_DATASET, poll_s=600,
                                      patterns=("preds_partial.tsv",), required="preds_partial.tsv").start()
        print("preds_partial.tsv is uploaded to", PREDS_DATASET)
    except Exception as exc:
        print("predictions are NOT uploaded (no KAGGLE_API_TOKEN secret?):", exc)

start = time.time()
print(len(todo), "lines to predict", flush=True)
try:
    with ThreadPoolExecutor(len(predictors)) as pool:
        futures = [pool.submit(run, k) for k in range(len(predictors))]
        while not all(f.done() for f in futures):
            time.sleep(60)
            done = sum(progress)
            speed = done / (time.time() - start)
            eta = (len(todo) - done) / speed / 60 if speed else float("nan")
            print(f"{time.strftime('%H:%M:%S')} {done} / {len(todo)} lines, {speed:.0f} lines/s, ~{eta:.0f} min left", flush=True)
finally:
    if uploader is not None:
        uploader.stop()  # uploads the last predictions, also after an error
        print("uploads to", PREDS_DATASET, ":", uploader.uploads)
for f in futures:
    f.result()  # raises the error of a failed predictor
assert len(preds) == len(rows), f"{len(rows) - len(preds)} lines without a prediction"
print(f"done in {(time.time() - start) / 60:.1f} min")

## 5. Disagreements

In [ ]:
import csv
import pandas as pd
from rapidfuzz.distance import Levenshtein

df = pd.DataFrame([(p, label, *preds[i]) for i, (p, label) in enumerate(rows)], columns=["path", "label", "pred", "score"])
df["dist"] = [Levenshtein.distance(a, b) for a, b in zip(df.label, df.pred)]
# dist_ns: edits ignoring spaces -- the text layer often splits words ("п р и я ти я"); such lines have dist_ns 0
df["dist_ns"] = [Levenshtein.distance(a.replace(" ", ""), b.replace(" ", "")) for a, b in zip(df.label, df.pred)]
df.to_csv("/kaggle/working/preds_all.tsv.gz", sep="\t", index=False, quoting=csv.QUOTE_NONE)
diff = df[df.dist_ns > 0].sort_values(["dist_ns", "score"], ascending=False)
spaces = df[(df.dist > 0) & (df.dist_ns == 0)].sort_values(["dist", "score"], ascending=False)
diff.to_csv("/kaggle/working/review.tsv", sep="\t", index=False, quoting=csv.QUOTE_NONE)
spaces.to_csv("/kaggle/working/review_spaces.tsv", sep="\t", index=False, quoting=csv.QUOTE_NONE)
print(f"{len(df)} lines, exact {(df.dist == 0).mean():.4f}, differ in characters {len(diff)}, only in spaces {len(spaces)}")
print(diff.groupby(pd.cut(diff.dist_ns, [0, 1, 2, 5, 10, 20, 1000]), observed=False).size().rename("lines by edits without spaces"))
print(diff.groupby(pd.cut(diff.score, [0, 0.5, 0.8, 0.9, 0.95, 1.0]), observed=False).size().rename("lines by score"))
pd.set_option("display.max_colwidth", 80)
diff.head(30)[["label", "pred", "score", "dist_ns"]]